# Patch extraction and reflective padding

This notebook confirms that `Patcher.extract` returns the correct sub-window for a given patch
index and that boundary patches are completed with reflective (symmetric) padding rather than
zeros. We compare reflective and constant padding modes side by side.

Modules exercised:

- `pipelines.dataset_pipeline.spatial.Patcher`

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy             as np
import matplotlib.pyplot as plt

import matplotlib
matplotlib.rcParams["figure.dpi"] = 110
matplotlib.rcParams["image.interpolation"] = "nearest"

SEED = 0
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
except Exception:
    pass

print("repo root on path:", REPO_ROOT)



## Labelled canvas

We use a gradient image so that any reflection at the border is visible as a mirrored
continuation of the intensity ramp.


In [ ]:
from pipelines.dataset_pipeline.spatial import Patcher

H, W = 40, 40
yy, xx = np.mgrid[0:H, 0:W]
canvas = (yy + xx).astype(np.float32)
canvas = canvas[np.newaxis]
print("canvas shape:", canvas.shape)

plt.figure(figsize=(4, 4))
plt.imshow(canvas[0], cmap="viridis")
plt.title("source canvas (azimuth + range gradient)")
plt.colorbar(fraction=0.046)
plt.tight_layout()
plt.show()



## Interior patch versus corner patch

We build a grid with padding and extract one interior patch and one corner patch. The corner
patch is the one that requires padding; its index is found from the per-patch coordinate list.


In [ ]:
patch_size = (16, 16)
stride     = 16

reflective = Patcher.build(spatial_size=(H, W), patch_size=patch_size, stride=stride, use_reflective_padding=True)
constant   = Patcher.build(spatial_size=(H, W), patch_size=patch_size, stride=stride, use_reflective_padding=False)

padded_indices   = [i for i, c in enumerate(reflective._patch_coords) if c[4] is not None]
interior_indices = [i for i, c in enumerate(reflective._patch_coords) if c[4] is None]
print("padded patch indices  :", padded_indices)
print("interior patch indices:", interior_indices)

corner_idx   = padded_indices[0]
interior_idx = interior_indices[0] if interior_indices else 0


In [ ]:
patch_interior = reflective.extract(canvas, interior_idx)
patch_reflect  = reflective.extract(canvas, corner_idx)
patch_constant = constant.extract(canvas, corner_idx)

fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
for ax, data, title in zip(
    axes,
    [patch_interior[0], patch_reflect[0], patch_constant[0]],
    [f"interior patch {interior_idx}", f"corner patch {corner_idx}\nreflective", f"corner patch {corner_idx}\nconstant (zero) pad"],
):
    im = ax.imshow(data, cmap="viridis")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()



## Reflective padding preserves continuity

For a smooth gradient, reflective padding mirrors the values across the border while constant
padding inserts a block of zeros. We plot a single row crossing the padded boundary.


In [ ]:
row = 0
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(patch_reflect[0, row],  "o-", label="reflective")
ax.plot(patch_constant[0, row], "s-", label="constant (zero)")
ax.set_xlabel("column within patch"); ax.set_ylabel("value")
ax.set_title(f"corner patch row {row}: padded region behaviour")
ax.legend()
plt.tight_layout()
plt.show()


## Expected visual outcome

The interior patch should reproduce a clean local crop of the gradient. The reflective corner
patch should show the gradient mirrored smoothly across the padded border, while the constant
corner patch should show an abrupt block of zeros. The row plot should make the zero-block of
constant padding clearly distinct from the mirrored values of reflective padding.